# 第23章 从随机游走到布朗运动——随机过程的直觉

> **动机先行**: 第19-22章我们研究的是真实价格序列的统计性质——平稳性、自相关、波动率聚集。但一个更根本的问题悬而未决: 价格序列为什么长这样? 它背后有没有一个简单的**生成机制**? 本章给出量化金融最重要的答案之一: 对数价格近似是一个**随机游走 (Random Walk)**, 而"无限细分后的随机游走"正是数学中最著名的随机过程——**布朗运动 (Brownian Motion)**。从掷硬币出发走到衍生品定价的地基, 只需要三步。
>
> **量化实战定位**: 布朗运动是期权定价、蒙特卡洛模拟、风险价值计算共同的语言基础。本章你将亲手模拟随机游走与几何布朗运动, 用 A 股真实数据检验 √t 法则, 并理解为什么"波动率拖累"会让高波动资产的典型收益反而更低。

---

## 23.1 动机: 股价为什么像醉汉走路

打开任意一只股票的日K线图, 你看到的都是锯齿状、无规则、事后解释头头是道、事前却难以预测的曲线。第19章我们用数据确认了两个事实:

1. **对数价格序列是非平稳的**——它没有固定的均值可以"回归", 因此用确定性函数去拟合它注定失败;
2. **对数收益率却近似平稳**——它总是在某个均值附近抖动。

把这两个事实拼起来, 会得到一个朴素得惊人结论: **价格是收益率的累加** (第5章的时间可加性), 而如果每天的收益率近似于一次"随机的硬币抛掷", 那么价格就是一条由硬币驱动的轨迹——**随机游走**。"醉汉走路"这个比喻来自统计学家 Karl Pearson 1905 年的论文: 一个醉汉每一步走多远、朝哪边都纯属随机, 走了 t 步之后离酒馆多远?

本章的任务是把这句直觉锻造成精确的数学:

- 第一步: 定义随机游走, 并算出它的均值和方差 (23.2节);
- 第二步: 让步子越来越碎、频率越来越高, 看它的极限——布朗运动 (23.3-23.4节);
- 第三步: 给布朗运动装上漂移和波动率, 得到股价模型的第一块基石——几何布朗运动 (23.5节);
- 最后回到真实市场: 随机游走假说在 A 股到底成立到什么程度? (23.6节)

在高中, 你熟悉的是确定性递推 $a_{n+1} = a_n + d$。随机游走只改了一个字: 把固定增量 $d$ 换成随机增量 $\varepsilon_n$。就是这一个字的改动, 打开了整个衍生品定价世界的大门。

---

## 23.2 简单随机游走——一切随机过程的起点

### 23.2.1 定义与金融对应

**定义 (简单随机游走)**: 设 $\varepsilon_1, \varepsilon_2, \dots$ 是独立同分布的随机变量,

$$
\varepsilon_t = \begin{cases} +1, & p = 1/2 \\ -1, & p = 1/2 \end{cases}
$$

称 $X_0 = 0$, $X_t = X_{t-1} + \varepsilon_t$ 为简单随机游走。等价地,

$$
X_t = \varepsilon_1 + \varepsilon_2 + \cdots + \varepsilon_t
$$

金融对应一目了然。第5章证明了对数收益率的**时间可加性**: $\ln P_t = \ln P_0 + r_1 + r_2 + \cdots + r_t$。如果把每天的对数收益率看作一个零均值的随机数, 那么对数价格就是一个步长连续版本的随机游走。这就是著名的**随机游走假说**——价格的历史路径不包含任何可用于预测未来的信息。

### 23.2.2 方差的线性增长与 √t 法则

利用期望的线性性和独立性 (第9章):

$$
E[X_t] = \sum_{i=1}^{t} E[\varepsilon_i] = 0
$$

$$
\mathrm{Var}(X_t) = \sum_{i=1}^{t} \mathrm{Var}(\varepsilon_i) = t \cdot 1 = t
$$

注意方差公式的关键前提: **独立性**。因为各步互不相关, 协方差项全部为零, 方差才能简单相加。

由此得到全书反复出现的 **√t 法则**:

$$
\mathrm{SD}(X_t) = \sqrt{t}
$$

**方差与时间成正比, 标准差只与时间的平方根成正比。** 金融意义立竿见影: 若日波动率为 1%, 一年 252 个交易日的波动率不是 $252\%$, 而是 $1\% \times \sqrt{252} \approx 15.9\%$。不确定性随时间增长, 但远比线性增长慢——这正是波动率年化公式 $\sigma_{年} = \sigma_{日}\sqrt{252}$ 的来源。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['font.sans-serif'] = ['WenQuanYi Micro Hei']
plt.rcParams['axes.unicode_minus'] = False

np.random.seed(42)

n_steps = 250          # 一年约250个交易日
n_paths = 2000         # 模拟路径数

# 每一步掷一次硬币: +1 或 -1, 概率各 1/2
steps = np.random.choice([-1, 1], size=(n_paths, n_steps))
walks = np.cumsum(steps, axis=1)      # 位置 = 步的累加

print("=== 随机游走的均值与标准差: 理论 vs 模拟 ===")
print(f"{'时刻 t':>8} | {'理论均值 0':>10} | {'模拟均值':>10} | {'理论std sqrt(t)':>14} | {'模拟std':>10}")
for t in [25, 50, 100, 250]:
    emp_mean = walks[:, t-1].mean()
    emp_std = walks[:, t-1].std()
    print(f"{t:>10} | {0.0:>10.4f} | {emp_mean:>10.4f} | {np.sqrt(t):>16.4f} | {emp_std:>10.4f}")

# 可视化: 前20条路径与 ±sqrt(t) 包络线
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
t_axis = np.arange(1, n_steps + 1)

axes[0].plot(t_axis, walks[:20].T, linewidth=0.9, alpha=0.8)
axes[0].plot(t_axis, np.sqrt(t_axis), 'k--', linewidth=2, label='+√t')
axes[0].plot(t_axis, -np.sqrt(t_axis), 'k--', linewidth=2, label='-√t')
axes[0].set_xlabel('步数 t', fontsize=12)
axes[0].set_ylabel('位置', fontsize=12)
axes[0].set_title('简单随机游走: 20条路径与±√t包络线', fontsize=13)
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

axes[1].plot(t_axis, walks.std(axis=0), 'b-', linewidth=2,
             label='模拟标准差 (2000条路径)')
axes[1].plot(t_axis, np.sqrt(t_axis), 'r--', linewidth=2, label='理论 √t')
axes[1].set_xlabel('步数 t', fontsize=12)
axes[1].set_ylabel('标准差', fontsize=12)
axes[1].set_title('不确定性按 √t 增长', fontsize=13)
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

**运行结果**:

```
=== 随机游走的均值与标准差: 理论 vs 模拟 ===
    时刻 t |     理论均值 0 |       模拟均值 |  理论std sqrt(t) |      模拟std
        25 |     0.0000 |    -0.2020 |           5.0000 |     4.9590
        50 |     0.0000 |    -0.0940 |           7.0711 |     7.0427
       100 |     0.0000 |    -0.2430 |          10.0000 |     9.9967
       250 |     0.0000 |     0.1020 |          15.8114 |    15.8680
```

![简单随机游走: 左图-20条路径与±√t包络线, 右图-2000条路径的标准差与理论√t几乎重合](images/ch23_fig1_random_walk_envelope.png)

**观察**:

1. **标准差与 √t 几乎完美吻合**: 四个时刻上, 2000 条路径的经验标准差 (4.96, 7.04, 10.00, 15.87) 与理论值 (5.00, 7.07, 10.00, 15.81) 的偏差都在 1% 以内。
2. **均值本身也是随机的**: 模拟均值在 0 附近小幅摆动 (-0.24 到 0.10)。这不是错误——经验均值是一个估计量, 它自己也有标准误 (第10章的语言: 估计量 ≈ 真值 ± 标准误)。

---

## 23.3 缩放极限——无限细分时随机游走变成了什么

### 23.3.1 思想实验: 为什么缩放因子必须是 √n

股价的演化不是一年只掷 250 次硬币——每一笔成交都在改变价格。把一年的区间 $[0, 1]$ 细分成 $n$ 段, 每段掷一次硬币。为了让极限过程有意义, 步幅必须同步缩小: 第 $i$ 步取 $\pm 1/\sqrt{n}$。定义缩放后的随机游走

$$
B^{(n)}_t = \frac{\varepsilon_1 + \varepsilon_2 + \cdots + \varepsilon_{\lfloor nt \rfloor}}{\sqrt{n}}
$$

为什么除以 $\sqrt{n}$ 而不是 $n$? 答案是**方差守恒**:

$$
\mathrm{Var}(B^{(n)}_1) = n \cdot \frac{1}{n} = 1
$$

无论细分得多细, 一年后总的不确定性保持不变。市场不会因为交易变频繁而变得没有风险。若除以 $n$, 方差会坍缩到 0——极限是一条静止的直线, 毫无意义。

**Donsker 不变原理** (非正式陈述): 只要每一步独立同分布、均值为 0、方差为 1 (具体是什么分布无关紧要——这就是"不变"二字的含义), 当 $n \to \infty$ 时, 整条缩放路径 $B^{(n)}_\cdot$ 都收敛到同一个随机过程——**布朗运动** $B_t$。

它与第10章中心极限定理的关系值得说清: 取定时刻 $t$, $B^{(n)}_t$ 的分布收敛到正态分布, 这正是 CLT 的内容; Donsker 定理的新信息是, 不仅每个时刻的分布收敛, **整条路径作为一个整体**也收敛。

### 23.3.2 模拟验证: 从二项到正态

$t = 1$ 处的终点值有个干净的解析形式: 记 $K_n$ 为 $n$ 次掷硬币中 "+1" 出现的次数, 则 $K_n \sim \mathrm{Binomial}(n, 1/2)$ (第8章二项分布), 且

$$
B^{(n)}_1 = \frac{2K_n - n}{\sqrt{n}}
$$

下面的代码模拟四种细分程度下的终点分布, 观察它们如何一步步"变成"正态分布。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

plt.rcParams['font.sans-serif'] = ['WenQuanYi Micro Hei']
plt.rcParams['axes.unicode_minus'] = False

np.random.seed(7)

m = 100000   # 每个设置下模拟的路径数

results = {}
print("=== 缩放后的终点分布 -> 标准正态 N(0,1) ===")
print(f"{'步数 n':>8} | {'均值':>8} | {'标准差':>8} | {'偏度':>8} | {'超额峰度':>8}")
for n in [5, 30, 100, 10000]:
    # 终点 = (2K - n)/sqrt(n), K 为 n 次掷硬币中 "+1" 的次数 ~ Binomial(n, 1/2)
    K = np.random.binomial(n, 0.5, m)
    endpoints = (2 * K - n) / np.sqrt(n)
    results[n] = endpoints
    skew = ((endpoints - endpoints.mean())**3).mean() / endpoints.std()**3
    exkurt = ((endpoints - endpoints.mean())**4).mean() / endpoints.std()**4 - 3
    print(f"{n:>8} | {endpoints.mean():>8.4f} | {endpoints.std():>8.4f} | {skew:>8.4f} | {exkurt:>8.4f}")

# 可视化: n=5 与 n=10000 的终点分布 vs 正态密度
fig, ax = plt.subplots(figsize=(10, 6))
x = np.linspace(-4.5, 4.5, 400)
ax.hist(results[5], bins=np.arange(-4.5, 4.6, 0.25), density=True,
        alpha=0.55, color='#FF9800', label='n=5 的终点分布')
ax.hist(results[10000], bins=np.arange(-4.5, 4.6, 0.15), density=True,
        alpha=0.55, color='#2196F3', label='n=10000 的终点分布')
ax.plot(x, stats.norm.pdf(x), 'k-', linewidth=2.5, label='标准正态密度 φ(x)')
ax.set_xlabel('缩放后的终点值', fontsize=12)
ax.set_ylabel('概率密度', fontsize=12)
ax.set_title('缩放极限: 步数越多, 终点分布越接近正态', fontsize=13)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

**运行结果**:

```
=== 缩放后的终点分布 -> 标准正态 N(0,1) ===
    步数 n |       均值 |      标准差 |       偏度 |     超额峰度
       5 |  -0.0003 |   1.0006 |  -0.0023 |  -0.4122
      30 |   0.0053 |   0.9978 |  -0.0010 |  -0.0453
     100 |  -0.0001 |   1.0015 |  -0.0037 |  -0.0168
   10000 |   0.0035 |   0.9996 |   0.0130 |  -0.0122
```

![缩放极限: n=5时终点分布仍是粗糙的两点混合(橙), n=10000时已与标准正态密度(黑线)几乎重合(蓝)](images/ch23_fig2_scaling_limit_normal.png)

**观察**: 偏度和超额峰度随 $n$ 增大迅速趋向 0。$n=5$ 时超额峰度为 $-0.41$ (两点分布的明显痕迹); 到 $n=10000$ 只剩 $-0.01$。正态分布不是假设出来的, 而是从掷硬币里"长"出来的。

但要强调: 终点分布只是布朗运动在**固定时刻的一个切片**。真正让布朗运动与众不同的是整条路径的形状——下一节的主角。

---

## 23.4 布朗运动——定义、性质与"粗糙度"

### 23.4.1 定义: 四条性质

**定义 (标准布朗运动 / 维纳过程)**: 随机过程 $\{B_t\}_{t \geq 0}$ 若满足以下四条, 则称为标准布朗运动:

1. **起点**: $B_0 = 0$
2. **独立增量**: 对不相交的时间区间, 增量 $B_{t_2}-B_{t_1}$ 与 $B_{t_4}-B_{t_3}$ 相互独立
3. **正态增量**: $B_t - B_s \sim N(0,\ t-s)$, 对任意 $0 \le s < t$
4. **路径连续**: $B_t$ 关于 $t$ 连续

三条直接推论 (由性质 2、3 与第8、9章的正态矩公式):

$$
E[B_t] = 0, \qquad \mathrm{Var}(B_t) = t, \qquad \mathrm{Cov}(B_s, B_t) = \min(s, t)
$$

协方差公式的推导只要一行, 却展示了独立增量的威力——把 $B_t$ 拆成 $B_s + (B_t - B_s)$:

$$
\mathrm{Cov}(B_s, B_t) = \mathrm{Var}(B_s) + \underbrace{\mathrm{Cov}(B_s,\ B_t - B_s)}_{=\,0\ (\text{独立增量})} = s
$$

**金融解读**: 两个时刻价格的联动程度, 由较早的那个时刻决定——共同走过的路才是相关的来源, 未来还没走的路与过去无关。

### 23.4.2 性质的金融解读

- **无记忆性 (马尔可夫性)**: 给定当前价格, 未来的分布与历史路径无关。这正是弱式有效市场的数学化身, 也与第19章"收益率自相关近似为零"的实证结果相互印证。
- **处处连续, 却处处不可导**: 放大任何一段路径, 看到的不是越来越平直的线段, 而是**同样粗糙的锯齿**。"瞬时速度"在任何时刻都不存在——第3章精心建立的导数工具, 在这里整体失效。这不是数学家的怪癖, 而是对"新信息随时到达、且到达本身无规律"的最忠实建模。
- **自相似性**: 图中的三连放大展示, 无论放大多少倍, 路径的视觉复杂度不变。

### 23.4.3 二次变差: 布朗运动的"指纹"

路径不可导并不意味着无法刻画它的波动。把 $[0, T]$ 分成 $n$ 小段, 考察增量平方和:

$$
\sum_{i=1}^{n} \left(\Delta B_i\right)^2 \xrightarrow{\ n \to \infty\ } T
$$

这是全章最惊人的事实: **增量本身完全随机, 但它们的平方之和收敛到一个确定的数**。直觉来自 $E[(\Delta B)^2] = \Delta t$: 每小段的平方增量平均而言恰好等于该段的时长, 加总即为 $T$。

这催生了一条在随机分析中无处不在的启发式记法:

$$
(\mathrm{d}B)^2 = \mathrm{d}t
$$

含义: 无穷小随机增量的平方**不是**可以被忽略的二阶小量, 而是**一阶小量**——与 $\mathrm{d}t$ 同阶。普通微积分中 $(dx)^2$ 可以放心丢弃, 在随机世界中不行。这就是后续章节伊藤引理中"泰勒展开必须多保留一项 $\Gamma$"的根本原因 (第6章债券凸性与第3章期权 Gamma 的随机版本)。

量化落地: 波动率交易中每天被买卖的**已实现方差 (Realized Variance)**, 正是日内高频收益率平方和——二次变差的实证版本。

### 23.4.4 伊藤等距初见

仿照黎曼和, 随机积分定义为 $\int_0^T f(t)\,\mathrm{d}B_t := \lim \sum f(t_i)\left(B_{t_{i+1}} - B_{t_i}\right)$。它满足一条优雅的方差恒等式——**伊藤等距**:

$$
E\left[\left(\int_0^T f(t)\,\mathrm{d}B_t\right)^2\right] = E\left[\int_0^T f^2(t)\,\mathrm{d}t\right]
$$

取最简单的 $f \equiv 1$: 左边是 $E[(B_T - B_0)^2] = T$, 右边是 $\int_0^T dt = T$, 恰好回到方差公式。直觉: 随机积分的"能量"由 $f^2$ 对时间的累积决定, 与 $f$ 本身的正负无关。本书将在随机微积分的核心章节用它控制期权定价公式中的各项方差; 本节只需记住这条恒等式的形式和特例。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['font.sans-serif'] = ['WenQuanYi Micro Hei']
plt.rcParams['axes.unicode_minus'] = False

np.random.seed(123)
n_paths = 5000      # 模拟路径数
n_steps = 1000      # [0,1] 区间的细分步数
T = 1.0
dt = T / n_steps

# 布朗运动的标准构造: 独立正态增量, 方差与时间步长成正比
dW = np.random.normal(0.0, np.sqrt(dt), size=(n_paths, n_steps))
W = np.cumsum(dW, axis=1)

def W_at(t):
    return W[:, int(t * n_steps) - 1]

print("=== 布朗运动的方差、协方差与二次变差: 理论 vs 模拟 ===")
print(f"{'统计量':>26} | {'理论值':>8} | {'模拟值':>8}")
print("-" * 52)
for t in [0.25, 0.5, 1.0]:
    print(f"{f'Var(B({t}))':>26} | {t:>8.4f} | {W_at(t).var():>8.4f}")
cov = np.cov(W_at(0.25), W_at(0.5))[0, 1]
print(f"{'Cov(B(0.25), B(0.5))':>26} | {0.25:>8.4f} | {cov:>8.4f}")

qv = (dW ** 2).sum(axis=1)
print(f"{'二次变差 sum(dW^2)':>24} | {T:>8.4f} | {qv.mean():>8.4f}")
print(f"\n二次变差的路径间标准差 = {qv.std():.4f}  (5000 条路径几乎给出同一个值)")

# 可视化: 同一条布朗运动路径在不同放大倍数下的形态 (另取一组冲击用于展示)
np.random.seed(99)
N_zoom = 100000
dz = np.random.normal(0.0, np.sqrt(1.0 / N_zoom), N_zoom)
Wz = np.concatenate([[0.0], np.cumsum(dz)])
tz = np.linspace(0.0, 1.0, N_zoom + 1)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
m1 = (tz >= 0.30) & (tz <= 0.32)
m2 = (tz >= 0.30) & (tz <= 0.3002)
axes[0].plot(tz, Wz, lw=0.6, color='#2196F3')
axes[0].set_title('B(t), t∈[0,1]', fontsize=12)
axes[1].plot(tz[m1], Wz[m1], lw=0.7, color='#E91E63')
axes[1].set_title('放大: t∈[0.30, 0.32]', fontsize=12)
axes[2].plot(tz[m2], Wz[m2], lw=0.7, color='#4CAF50')
axes[2].set_title('再放大100倍: t∈[0.3000, 0.3002]', fontsize=12)
for ax in axes:
    ax.grid(True, alpha=0.3)
    ax.set_xlabel('t', fontsize=11)
axes[0].set_ylabel('B(t)', fontsize=11)
plt.tight_layout()
plt.show()

**运行结果**:

```
=== 布朗运动的方差、协方差与二次变差: 理论 vs 模拟 ===
                       统计量 |      理论值 |      模拟值
----------------------------------------------------
              Var(B(0.25)) |   0.2500 |   0.2539
               Var(B(0.5)) |   0.5000 |   0.5286
               Var(B(1.0)) |   1.0000 |   1.0180
      Cov(B(0.25), B(0.5)) |   0.2500 |   0.2596
          二次变差 sum(dW^2) |   1.0000 |   1.0004

二次变差的路径间标准差 = 0.0444  (5000 条路径几乎给出同一个值)
```

![布朗运动的粗糙度: 同一路径从 t 属于区间 0 到 1, 放大到 0.30-0.32, 再放大到 0.3000-0.3002, 锯齿形态始终不变——处处连续、处处不可导](images/ch23_fig3_brownian_roughness.png)

**观察**:

1. **方差与协方差结构全部吻合**: Var 与 Cov 的模拟值贴近理论值 (协方差 0.2596 vs 0.25, 差异属于 5000 条路径的抽样噪声)。
2. **二次变差是"确定"的随机量**: 5000 条彼此独立的路径, 平方增量之和全都落在 1.000 附近 (路径间标准差只有 0.044), 且随细分步数增加进一步收缩——每条路径都"撞向"同一个极限 $T$。

---

## 23.5 几何布朗运动——股价模型的第一块基石

### 23.5.1 为什么不能直接拿布朗运动当股价

布朗运动虽然优美, 直接当股价模型有三处硬伤:

1. **会变负**: 正态增量可加, 价格迟早穿到 0 以下; 而股票有有限责任, 价格恒非负。
2. **加法错了对象**: 投资者关心百分比收益——100 元的股票涨 10 元是 +10%, 10 元的股票涨 10 元是 +100%。绝对增量相同, 金融意义天壤之别。
3. **可加的量选错了**: 第5章告诉我们, 跨期可加的是**对数收益率**, 不是价格增量。所以"随机游走"应该发生在 $\ln S$ 上, 而不是 $S$ 上。

### 23.5.2 从对数价格到几何布朗运动及其解

顺着第 3 点, 设对数价格是带漂移的布朗运动:

$$
\ln S_T = \ln S_0 + \left(\mu - \frac{\sigma^2}{2}\right)T + \sigma B_T
$$

两边取指数, 得到**几何布朗运动 (Geometric Brownian Motion, GBM)** 的解:

$$
S_T = S_0 \exp\left(\left(\mu - \frac{\sigma^2}{2}\right)T + \sigma B_T\right)
$$

其微分形式 (SDE) 就是衍生品定价书里的标准开场:

$$
\mathrm{d}S_t = \mu S_t\,\mathrm{d}t + \sigma S_t\,\mathrm{d}B_t
$$

参数解读: $\mu$ 是期望年化对数收益 (**漂移**, drift), 决定路径的平均去向; $\sigma$ 是年化波动率 (**扩散**, diffusion), 决定路径的抖动幅度。

那个神秘的 $-\sigma^2/2$ 从哪来? 两个视角:

- **代数视角**: 第5章的泰勒展开 $\ln(1+x) \approx x - x^2/2$。先涨后跌的不对称意味着平均增长被波动拉低一截;
- **随机视角**: $(\mathrm{d}B)^2 = \mathrm{d}t$ (23.4.3节)! 泰勒展开的二阶项与一阶项同阶, 必须保留。后续章节的伊藤引理将把这行直觉变成严格的推导。

由于 $\ln S_T$ 服从正态分布, $S_T$ 服从**对数正态分布** (第8章)——永远为正, 三处硬伤一并解决。

### 23.5.3 波动率拖累: 高波动的隐形税

对数正态分布有一个让初学者意外的性质: **期望与中位数分离**。

$$
E[S_T] = S_0 e^{\mu T}, \qquad \text{中位数}(S_T) = S_0 e^{(\mu - \sigma^2/2)T}
$$

期望只由 $\mu$ 决定, 与 $\sigma$ 无关; 中位数却被 $\sigma$ 征了一道税。下面的蒙特卡洛实验让这道税现出原形。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['font.sans-serif'] = ['WenQuanYi Micro Hei']
plt.rcParams['axes.unicode_minus'] = False

np.random.seed(2024)

S0 = 100.0     # 初始价格
mu = 0.08      # 年化漂移 8%
T = 1.0        # 期限 1 年
n_steps = 250  # 一年250个交易日
dt = T / n_steps
n_mc = 100000  # 蒙特卡洛路径数

Z = np.random.standard_normal((n_mc, n_steps))   # 共享同一组标准化冲击

print("=== 几何布朗运动: 波动率 sigma 对一年后股价分布的影响 ===")
print(f"参数: S0={S0:.0f}, mu={mu*100:.0f}%/年, T={T:.0f}年, MC路径数={n_mc}")
print(f"{'sigma':>7} | {'MC均值':>9} | {'理论E[S_T]':>10} | {'MC中位数':>9} | {'理论中位数':>10} | {'P(S_T>S0)':>9}")
for sigma in [0.10, 0.20, 0.40]:
    log_ret = (mu - 0.5*sigma**2)*dt + sigma*np.sqrt(dt)*Z
    ST = S0 * np.exp(log_ret.sum(axis=1))
    mean_theo = S0 * np.exp(mu * T)
    med_theo = S0 * np.exp((mu - 0.5*sigma**2) * T)
    p_win = (ST > S0).mean()
    print(f"{sigma:>9.2f} | {ST.mean():>9.2f} | {mean_theo:>10.2f} | "
          f"{np.median(ST):>9.2f} | {med_theo:>10.2f} | {p_win:>11.4f}")

# 可视化: 三种波动率共用同一组随机冲击的样本路径
Z_show = np.random.standard_normal((3, n_steps))   # 展示用的另一组冲击
t_axis = np.linspace(0, T, n_steps + 1)
fig, ax = plt.subplots(figsize=(11, 6))
colors = ['#4CAF50', '#2196F3', '#E91E63']
for sigma, c in zip([0.10, 0.20, 0.40], colors):
    increments = (mu - 0.5*sigma**2)*dt + sigma*np.sqrt(dt)*Z_show[0]
    log_path = np.concatenate([[0.0], np.cumsum(increments)])
    S_path = S0 * np.exp(log_path)
    med_T = S0 * np.exp((mu - 0.5*sigma**2) * T)
    ax.plot(t_axis, S_path, color=c, linewidth=1.8,
            label=f'σ={sigma*100:.0f}%: 一年后中位数={med_T:.1f}')
ax.axhline(S0, color='gray', linestyle='--', linewidth=1)
ax.text(0.01, S0 + 1.0, '$S_0$=100', fontsize=10, color='gray')
ax.set_xlabel('时间 t (年)', fontsize=12)
ax.set_ylabel('股价 $S_t$', fontsize=12)
ax.set_title('几何布朗运动: 同一组随机冲击下, σ对路径的影响 (μ=8%)', fontsize=13)
ax.legend(fontsize=11, loc='upper left')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

**运行结果**:

```
=== 几何布朗运动: 波动率 sigma 对一年后股价分布的影响 ===
参数: S0=100, mu=8%/年, T=1年, MC路径数=100000
  sigma |      MC均值 |   理论E[S_T] |     MC中位数 |      理论中位数 | P(S_T>S0)
     0.10 |    108.36 |     108.33 |    107.79 |     107.79 |      0.7751
     0.20 |    108.40 |     108.33 |    106.19 |     106.18 |      0.6182
     0.40 |    108.48 |     108.33 |    100.01 |     100.00 |      0.5001
```

![几何布朗运动: 同一组随机冲击下σ=10%/20%/40%的三条路径, 波动率越高路径越狂野, 中位数终点越低](images/ch23_fig4_gbm_volatility_effect.png)

**观察**:

1. **期望纹丝不动**: 三种波动率下 MC 均值都停在 108.3 附近——期望确实只认 $\mu$, 不认 $\sigma$。
2. **中位数持续下沉**: 从 107.79 (σ=10%) 跌到 100.01 (σ=40%)。当 $\mu - \sigma^2/2 = 0.08 - 0.08 = 0$ 时, **典型路径一年后原地踏步**!
3. **胜率崩塌**: "跑赢初始价"的概率从 77.5% 掉到 50.0%。期望之所以还高, 全靠少数暴涨路径撑着。

这就是**波动率拖累 (volatility drag)**: 高波动资产的典型 (中位数) 收益系统性低于低波动资产, 尽管两者期望可能相同。它是第5章"算术平均 > 几何平均"在连续时间里的化身, 也是长期复利投资者对波动率深恶痛绝的数学根源。

---

## 23.6 随机游走假设与真实市场——A股证据

模型建好了, 该拿真实数据拷问它。三个问题:

1. 收益率真的近似不相关吗 (随机游走的核心预言)?
2. √t 法则在真实数据上成立吗?
3. 正态增量假设靠谱吗?

用贵州茅台 2023-05 至 2026-05 共 724 个交易日的高频对数收益率逐一检验。

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.stats.diagnostic import acorr_ljungbox

plt.rcParams['font.sans-serif'] = ['WenQuanYi Micro Hei']
plt.rcParams['axes.unicode_minus'] = False

csv_path = 'data/ifind_price_data.csv'
df = pd.read_csv(csv_path)
mt = df[df['thscode'] == '600519.SH'].sort_values('time').reset_index(drop=True)

lp = np.log(mt['close'])
r = lp.diff().dropna()

print("=== 真实市场检验: 贵州茅台 日对数收益率 ===")
print(f"样本区间: {mt['time'].iloc[0]} ~ {mt['time'].iloc[-1]}, 共 {len(mt)} 个交易日")

mu_d, sig_d = r.mean(), r.std()
print(f"日均值 = {mu_d*100:.4f}%,  日波动率 = {sig_d*100:.4f}%")
print(f"年化波动率 = {sig_d*np.sqrt(252)*100:.2f}%   (sqrt(252) 法则)")

acf1 = r.autocorr(1)
lb = acorr_ljungbox(r, lags=[10], return_df=True)
print(f"滞后1阶自相关 = {acf1:+.4f}, Ljung-Box Q(10) p值 = {lb['lb_pvalue'].iloc[0]:.4f}")
print("-> 自相关统计上显著, 但幅度很小(滞后1阶仅+5%): 随机游走是好的第一近似, 却非严格成立")

exkurt = r.kurt()
print(f"超额峰度 = {exkurt:.2f}  (正态分布为 0; 远大于 0 => 厚尾)")

print()
print("=== sqrt(k) 缩放检验: k日收益率的波动率 ===")
print(f"{'k(天)':>6} | {'实际std':>10} | {'sqrt(k)x日std':>14} | {'比值':>7}")
for k in [1, 2, 5, 10, 20]:
    rk = lp.iloc[::k].diff().dropna()
    actual = rk.std()
    pred = sig_d * np.sqrt(k)
    print(f"{k:>8} | {actual:>12.6f} | {pred:>16.6f} | {actual/pred:>9.4f}")

rb = acorr_ljungbox(np.abs(r - r.mean()), lags=[10], return_df=True)
print()
print(f"|r| 序列的 Ljung-Box Q(10) p值 = {rb['lb_pvalue'].iloc[0]:.2e}")
print("-> 收益率的自相关微弱, 但|r|的依赖极强: 价格近似随机游走, 波动率却明显聚集")

# 可视化: 左图-收益率与|收益率|的自相关对比, 右图-sqrt(k)缩放检验
def acf_vals(x, nlags):
    x = x - x.mean()
    return np.array([1.0] + [np.corrcoef(x[:-l], x[l:])[0, 1]
                             for l in range(1, nlags + 1)])

nlags = 15
lags = np.arange(nlags + 1)
acf_r = acf_vals(r.values, nlags)
acf_abs = acf_vals(np.abs(r.values), nlags)
band = 1.96 / np.sqrt(len(r))

ks = np.array([1, 2, 5, 10, 20])
actual = np.array([lp.iloc[::k].diff().dropna().std() for k in ks])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
w = 0.35
axes[0].bar(lags - w/2, acf_r, w, label='收益率 r', color='#2196F3', alpha=0.85)
axes[0].bar(lags + w/2, acf_abs, w, label='|收益率| |r|', color='#E91E63', alpha=0.85)
axes[0].axhline(band, color='gray', linestyle='--', linewidth=1)
axes[0].axhline(-band, color='gray', linestyle='--', linewidth=1)
axes[0].axhline(0, color='black', linewidth=0.8)
axes[0].set_xlabel('滞后阶数', fontsize=12)
axes[0].set_ylabel('自相关系数', fontsize=12)
axes[0].set_title('r 与 |r| 的自相关函数 (虚线为±1.96/√N)', fontsize=12)
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

axes[1].plot(ks, actual, 'o-', linewidth=2, markersize=8,
             label='实际 k 日波动率', color='#2196F3')
axes[1].plot(ks, sig_d*np.sqrt(ks), 's--', linewidth=2, markersize=8,
             label='√k 法则预测', color='#FF9800')
axes[1].set_xticks(ks)
axes[1].set_xlabel('持有期 k (天)', fontsize=12)
axes[1].set_ylabel('收益率标准差', fontsize=12)
axes[1].set_title('√k 缩放检验: 实际 vs 理论', fontsize=12)
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

**运行结果**:

```
=== 真实市场检验: 贵州茅台 日对数收益率 ===
样本区间: 2023-05-25 ~ 2026-05-22, 共 724 个交易日
日均值 = -0.0245%,  日波动率 = 1.4319%
年化波动率 = 22.73%   (sqrt(252) 法则)
滞后1阶自相关 = +0.0504, Ljung-Box Q(10) p值 = 0.0001
-> 自相关统计上显著, 但幅度很小(滞后1阶仅+5%): 随机游走是好的第一近似, 却非严格成立
超额峰度 = 8.06  (正态分布为 0; 远大于 0 => 厚尾)

=== sqrt(k) 缩放检验: k日收益率的波动率 ===
  k(天) |      实际std |   sqrt(k)x日std |      比值
       1 |     0.014319 |         0.014319 |    1.0000
       2 |     0.020718 |         0.020250 |    1.0231
       5 |     0.034497 |         0.032017 |    1.0775
      10 |     0.055355 |         0.045280 |    1.2225
      20 |     0.062113 |         0.064035 |    0.9700

|r| 序列的 Ljung-Box Q(10) p值 = 4.53e-23
-> 收益率的自相关微弱, 但|r|的依赖极强: 价格近似随机游走, 波动率却明显聚集
```

![贵州茅台随机游走检验: 左图-r与|r|的自相关函数(r基本贴着置信带,|r|全面突破), 右图-k日波动率与√k法则预测大体吻合](images/ch23_fig5_real_market_test.png)

**四条结论**:

1. **随机游走是好的第一近似, 但不是真理**: 滞后1阶自相关只有 +5%, Ljung-Box 检验却在 1% 水平上显著 (p=0.0001)——统计上可检测, 经济上难利用 (考虑交易成本后更难)。这点微弱的预测性正是多因子研究的土壤, 也是"市场有效但不必绝对有效"的注脚。
2. **√t 法则可用但不精确**: k 日波动率与 $\sqrt{k}$ 预测的比值在 0.99~1.23 之间。实务中年化波动率 $\times\sqrt{252}$ 完全够用; 偏离部分则携带信息——比值持续大于 1 提示正自相关 (趋势), 小于 1 提示均值回复。
3. **厚尾严重**: 超额峰度 8.06, 远超正态的 0。用正态假设管理极端风险会系统性地掉以轻心——第11章多重检验的教训在此同样适用。
4. **波动率自己在动**: 收益率 r 几乎不相关, 但 $|r|$ 的 Ljung-Box p 值是 $10^{-22}$ 量级! 大波动后面跟着大波动——这正是第21章 GARCH 模型的存在理由。换句话说: **价格近似随机游走, 条件波动率却绝非常数**。

一句话总结: 布朗运动是真实市场的零阶近似; 微弱自相关、厚尾、波动率聚集是叠加在这个骨架上的三类修正, 后续章节的模型都将围绕它们展开。

---

## 23.7 核心公式速查

> 本节是前述各节公式的集中汇总, 供复习和查阅使用.

1. **简单随机游走**: $X_t = X_{t-1} + \varepsilon_t$, $\varepsilon_t$ 独立同分布, $P(\pm 1)=1/2$
2. **前两阶矩**: $E[X_t] = 0$, $\mathrm{Var}(X_t) = t$ (独立性是方差相加的前提)
3. **√t 法则**: $\mathrm{SD}(X_t) = \sqrt{t}$; k日波动 $\approx \sqrt{k}\,\times$ 日波动; 年化 $\sigma_{年} = \sigma_{日}\sqrt{252}$
4. **缩放后的随机游走**: $B^{(n)}_t = (\varepsilon_1 + \cdots + \varepsilon_{\lfloor nt\rfloor})/\sqrt{n}$, 除以 $\sqrt{n}$ 保证方差守恒
5. **Donsker 不变原理**: $B^{(n)} \Rightarrow B_t$ ($n\to\infty$), 极限与步长分布无关; 固定时刻即 CLT
6. **布朗运动定义**: $B_0=0$; 独立增量; $B_t - B_s \sim N(0, t-s)$; 路径连续
7. **矩结构**: $E[B_t]=0$; $\mathrm{Var}(B_t)=t$; $\mathrm{Cov}(B_s, B_t)=\min(s,t)$
8. **二次变差**: $\sum_i (\Delta B_i)^2 \to T$ (确定值); 启发式记法 $(\mathrm{d}B)^2 = \mathrm{d}t$
9. **伊藤等距**: $E\big[(\int_0^T f\,\mathrm{d}B)^2\big] = E\big[\int_0^T f^2\,\mathrm{d}t\big]$; 特例 $\mathrm{Var}(B_T)=T$
10. **GBM 的解**: $S_T = S_0 \exp\big((\mu-\sigma^2/2)T + \sigma B_T\big)$; $\ln S_T \sim N\big(\ln S_0 + (\mu-\sigma^2/2)T,\ \sigma^2 T\big)$
11. **期望与中位数分离**: $E[S_T] = S_0 e^{\mu T}$, 中位数 $= S_0 e^{(\mu-\sigma^2/2)T}$ (波动率拖累)

## 23.8 本章小结

| 概念 | 核心要点 | 量化意义 |
|------|---------|---------|
| 随机游走 | 独立同分布步的累加 | 对数价格模型的零阶近似 |
| √t 法则 | 方差 ∝ t, 标准差 ∝ √t | 波动率年化与跨期换算的依据 |
| Donsker 不变原理 | 缩放随机游走的极限是布朗运动 | CLT 的路径版本; 正态性从离散长出 |
| 布朗运动 | 独立增量 + 正态增量 + 连续路径 | 衍生品定价的通用语言 |
| 二次变差 | $\sum(\Delta B)^2 \to T$ 为确定值 | 已实现方差的理论原型 |
| 伊藤等距 | 随机积分的方差恒等式 | 定价与风控中方差计算的钥匙 |
| 几何布朗运动 | $S_T = S_0 e^{(\mu-\sigma^2/2)T+\sigma B_T}$ | Black-Scholes 公式的模型前提 |
| 波动率拖累 | 中位数 < 期望 | 长期复利中波动的隐形成本 |

**最后一句话**: 本章我们从掷硬币出发, 一步一步走到了期权定价的地基。随机分析的大厦并不神秘——它只是把"每一步都是随机的"这句话, 用极限的语言讲到了底。

## 23.9 练习题

### 数学推导

**题1——随机游走的矩与位置分布**:

(a) 利用期望线性性与独立性, 证明 $E[X_t] = 0$ 且 $\mathrm{Var}(X_t) = t$。

(b) 记 $K_t$ 为前 $t$ 步中 "+1" 出现的次数, 证明 $X_t = 2K_t - t$, 并写出 $P(X_t = k)$ 的二项表达式。

(c) 计算 $t=3$ 时的 $P(X_3 = 1)$ 和 $P(|X_3| = 3)$。

**题2——缩放下方差的守恒**:

(a) 证明对任意 $n$, 缩放终点 $Y_n = S_n/\sqrt{n}$ 满足 $E[Y_n] = 0$ 且 $\mathrm{Var}(Y_n) = 1$。

(b) 如果改用 $Z_n = S_n/n$ 缩放, 求 $\lim_{n\to\infty} \mathrm{Var}(Z_n)$, 并解释为什么这种缩放得不到有意义的过程。

(c) 结合第10章的中心极限定理说明: $\lim_{n\to\infty} P(Y_n \le x) = \Phi(x)$。

**题3——GBM 的期望与中位数**:

设 $\ln S_T \sim N(m, v)$, 其中 $m = \ln S_0 + (\mu-\sigma^2/2)T$, $v = \sigma^2 T$。

(a) 利用正态分布的对称性证明 $S_T$ 的中位数为 $S_0 \exp((\mu-\sigma^2/2)T)$。

(b) 利用正态矩公式 $E[e^{\lambda Z}] = e^{\lambda^2/2}$ ($Z\sim N(0,1)$) 证明 $E[S_T] = S_0 e^{\mu T}$, 并说明为什么该结果与 $\sigma$ 无关。

(c) 当 $\mu = 8\%$, $\sigma = 40\%$, $T = 1$ 时, 典型路径 (中位数) 相对 $S_0$ 的涨跌幅是多少? 这解释了正文哪个数值现象?

### 编程实践

**题1——协方差结构的验证**: 模拟 20000 条布朗运动路径 (T=1, 1000 步)。对 $s, t \in \{0.1, 0.3, 0.5, 0.7, 1.0\}$ 的所有组合计算 $\mathrm{Cov}(B_s, B_t)$ 的模拟值, 并验证它与 $\min(s, t)$ 的相对误差不超过 5%。(提示: 参考本章 23.4.3 节代码中的 `np.cov` 用法。)

**题2——指数与个股的随机游走比较**: 用 `data/ifind_price_data.csv` 中的沪深300 (`thscode == '000300.SH'`) 重做 23.6 节的全部检验 (Ljung-Box、超额峰度、√k 缩放比值), 与贵州茅台的结果逐项对比。指数的随机游走假说成立得更好还是更差? 给出一个经济学解释。(提示: 指数是个股的组合, 分散化会削弱个股特质波动。)

## 23.10 参考文献

1. Ross, S. M. (1996). *Stochastic Processes* (2nd ed.). Wiley.（随机过程的经典入门教材, 其布朗运动一章的讲述节奏与本章最接近）

2. Shreve, S. E. (2004). *Stochastic Calculus for Finance II: Continuous-Time Models*. Springer.（用二叉树极限严格构造布朗运动——23.3节缩放极限的完整数学版）

3. Malkiel, B. G. (2019). *A Random Walk Down Wall Street* (12th ed.). W. W. Norton.（"漫步华尔街", 随机游走思想最有影响的大众读本）

4. Campbell, J. Y., Lo, A. W., & MacKinlay, A. C. (1997). *The Econometrics of Financial Markets*. Princeton University Press.（第2章系统讲解随机游走假说的各种计量检验, 23.6节的严谨版）